# Analyse du potentiel territorial d'implantation de bornes de recharge — France

**Objectif :** produire une *première analyse de potentiel territorial*. On repère des territoires **sous-équipés selon des critères documentés**,
puis on propose trois territoires à étudier plus finement.

**Sources**
- IRVE : fichier consolidé des infrastructures de recharge (data.gouv.fr / transport.data.gouv.fr).
- Nb d'habitants, communes et EPCI : INSEE / DGCL via le paquet `@etalab/decoupage-administratif` (COG 2026).
- Contours des communes: dépôt `gregoiredavid/france-geojson`.

**Unité territoriale :** EPCI à fiscalité propre (la Métropole du Grand Paris est éclatée en territoires plus petit, sinon elle écrase tout le reste).

In [ ]:
#on importe les bibliothèques nécessaires 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
import re
from pathlib import Path
from IPython.display import display
try:
    import folium
    FOLIUM_DISPONIBLE = True
except ImportError:
    FOLIUM_DISPONIBLE = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

#palette pastel pour les graphiqueS
PALETTE_PASTEL = ["#A8DADC", "#F4C2C2", "#CDB4DB", "#FFD6A5", "#BDE0FE", "#CAFFBF", "#FFAFCC", "#D8E2DC"]

DOSSIER = Path.cwd()
FICHIER_SOURCE = DOSSIER / "bornes-irve.csv"
DOSSIER_EXPORT = DOSSIER / "exports_irve"
DOSSIER_EXPORT.mkdir(exist_ok=True)

In [ ]:
df_source = pd.read_csv(FICHIER_SOURCE, low_memory=False)
df = df_source.copy()

nombre_lignes = f"{len(df):,}".replace(",", " ")
print("Nombre de lignes :", nombre_lignes)
print(f"Nombre de colonnes : {len(df.columns)}")
display(df.head()) #voir comment se présentent les données dans un tableau "propre"
display(pd.DataFrame({ "type": df.dtypes.astype(str),"valeurs_manquantes": df.isna().sum(),}).sort_values("valeurs_manquantes", ascending=False)) 

In [ ]:
print("Dimensions :", df.shape)
print("\nColonnes disponibles :",df.columns.tolist())
#print(df.columns.tolist())

colonnes_importantes = [
    "nom_station", "nom_enseigne", "implantation_station", "adresse_station",
    "code_insee_commune", "consolidated_commune", "coordonneesXY",
    "consolidated_longitude", "consolidated_latitude", "nbre_pdc",
    "id_pdc_itinerance", "id_station_itinerance", "puissance_nominale",
    "condition_acces", "date_mise_en_service"
]
display(df[[i for i in colonnes_importantes if i in df.columns]].head())

print("Lignes entièrement identiques :", int(df.duplicated().sum()))
print("Identifiants PDC manquants :", int(df["id_pdc_itinerance"].isna().sum()))
print("Identifiants station manquants :", int(df["id_station_itinerance"].isna().sum()))

In [ ]:
#nettoyage et normalisation des valeurs vides
colonnes_texte = df.select_dtypes(include=["object", "string"]).columns

valeurs_vides = ["", "NA", "N/A", "nan", "None", "NULL"]

for colonne in colonnes_texte:
    # On enlève les espaces
    df[colonne] = df[colonne].str.strip().replace(valeurs_vides, np.nan)

#on convertit tout ce qui n'est pas un nb
for colonne in [ "puissance_nominale","nbre_pdc", "consolidated_longitude","consolidated_latitude", "consolidated_code_postal",]:
    if colonne in df.columns:
        df[colonne] = pd.to_numeric(df[colonne], errors="coerce")

#on convertit les dates 
for colonne in ["date_mise_en_service", "date_maj", "last_modified", "created_at"]:
    if colonne in df.columns:
        df[colonne] = pd.to_datetime(df[colonne], errors="coerce", format="mixed")

print("Dimensions après nettoyage:", df.shape)

In [ ]:
def normaliser_texte(valeur):
    if pd.isna(valeur):
        return ""
    texte = str(valeur).lower().strip()
    texte = texte.replace("ŕ", "à").replace("ā", "à").replace("Ã ", "à")
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    texte = re.sub(r"\s+", " ", texte)
    return texte

def texte_ligne(ligne, colonnes):
    morceaux = []
    for c in colonnes:
        if c in ligne.index and pd.notna(ligne[c]):
            morceaux.append(str(ligne[c]))
    return " | ".join(morceaux)

def classer_implantation(ligne):
    source = normaliser_texte(ligne.get("implantation_station", ""))
    nom_adresse = normaliser_texte(texte_ligne(ligne, ["nom_station", "nom_enseigne", "adresse_station", "observations"]))


    if "voirie" in source or "voie publique" in source:
        return "Voie publique"
    if "parking prive reserve a la clientele" in source:
        return "Parking privé réservé à la clientèle"
    if "parking prive a usage public" in source:
        return "Parking privé à usage public"
    if "parking public" in source:
        return "Parking public"

#reclassement des lignes "Autre" / "Station dédiée" à partir d'indices explicites.
    indices_clientele = ["reserve a la clientele", "parking client", "stationnement client","clients uniquement", "clientele uniquement"]
    
    if any(m in nom_adresse for m in indices_clientele):
        return "Parking privé réservé à la clientèle"

    enseignes = ["leclerc", "carrefour", "intermarche", "intermarche", "lidl","auchan", "super u", "hyper u", "casino", "brico", "spar","centre commercial", "zone commerciale"]
    mots_parking = ["parking", "parc de stationnement", "centre commercial", "zone commerciale"]
    
    if any(m in nom_adresse for m in enseignes) and any(m in nom_adresse for m in mots_parking):
        return "Parking privé à usage public"

    if "parking prive" in nom_adresse and any(m in nom_adresse for m in ["entreprise", "societe", "personnel", "employe"]):
        return "Parking privé réservé à la clientèle"

    if any(m in nom_adresse for m in ["parking public", "parking municipal", "parking mairie","parking de la mairie", "parking de la gare"]):
        return "Parking public"

    return "Autre"

df["implantation_source"] = df["implantation_station"]
df["implantation_categorie"] = df.apply(classer_implantation, axis=1)

ordre_implantations = ["Voie publique","Parking privé réservé à la clientèle","Parking privé à usage public","Parking public","Autre"]

print("Répartition après harmonisation et reclassement :")
display(df["implantation_categorie"].value_counts(dropna=False).reindex(ordre_implantations, fill_value=0).rename_axis("Catégorie").to_frame("Nombre de points de charge"))

print("Exemples encore classés 'Autre' :")

colonnes_possibles = ["nom_station", "nom_enseigne", "adresse_station", "implantation_source", "implantation_categorie"]
colonnes_exemples = []

for c in colonnes_possibles:
    if c in df.columns:
        colonnes_exemples.append(c)

filtre = df["implantation_categorie"] == "Autre"
display(df.loc[filtre, colonnes_exemples].head(10))

In [ ]:
#on convertit tout en kW
df["puissance_source"] = df["puissance_nominale"]
df["puissance_kW"] = df["puissance_nominale"]
masque_probablement_watts = df["puissance_nominale"] > 1000
df.loc[masque_probablement_watts, "puissance_kW"] = (df.loc[masque_probablement_watts, "puissance_nominale"] / 1000)
df["puissance_conversion_depuis_W"] = masque_probablement_watts

print("Valeurs de puissance manquantes :", int(df["puissance_kW"].isna().sum()))
print("Valeurs ayant fait l'objet de la conversion supposée W → kW :", int(masque_probablement_watts.sum()))

In [ ]:
def categorie_vitesse(p):
    if pd.isna(p) or p <= 0:
        return "Puissance inconnue ou nulle"
    elif p <= 7:
        return "Lente (≤ 7 kW)"
    elif p <= 22:
        return "Accélérée (> 7 à 22 kW)"
    elif p <= 50:
        return "Rapide (> 22 à 50 kW)"
    else:
        return "Très rapide (> 50 kW)"

df["categorie_vitesse"] = df["puissance_kW"].apply(categorie_vitesse)

display(df["categorie_vitesse"].value_counts().to_frame("Nombre de points de charge"))

In [ ]:
#supprimer les lignes strictement identiques.
nb_avant = len(df)
df = df.drop_duplicates().copy()

#détecter les ipc répétés, sans les supprimer automatiquement car les lignes peuvent porter des info différentes:on garde une table d'audit pour décider à part ce qu'on en fait (pas fait ici)
if "id_pdc_itinerance" in df.columns:
    masque_id_repete = df["id_pdc_itinerance"].notna() & df["id_pdc_itinerance"].duplicated(keep=False)
    doublons_pdc_a_verifier = df.loc[masque_id_repete].sort_values("id_pdc_itinerance").copy()
    print("Lignes portant un identifiant PDC répété :", len(doublons_pdc_a_verifier))
    print("Nb de bornes uniques concernées par les doublons :", doublons_pdc_a_verifier["id_pdc_itinerance"].nunique())
    colonnes_voulues = ["id_pdc_itinerance","id_station_itinerance","nom_station","adresse_station","consolidated_longitude","consolidated_latitude","puissance_kW","implantation_categorie"]
colonnes_a_afficher = []
for c in colonnes_voulues:
    if c in doublons_pdc_a_verifier.columns:
        colonnes_a_afficher.append(c)

display(doublons_pdc_a_verifier[colonnes_a_afficher].head(10))

In [ ]:
if "consolidated_longitude" in df.columns:
    df["longitude"] = pd.to_numeric(df["consolidated_longitude"], errors="coerce")
if "consolidated_latitude" in df.columns:
    df["latitude"] = pd.to_numeric(df["consolidated_latitude"], errors="coerce")

#on crée une commune d'analyse de secours (si le nom est vide, on met "Inconnue")
if "consolidated_commune" in df.columns:
    df["commune_analyse"] = df["consolidated_commune"].fillna("Commune inconnue").astype(str)
else:
    df["commune_analyse"] = "Commune inconnue"

#contrôle qualité : repérage des bornes hors de France métropolitaine pr éviter après que la carte s'affiche mal si coordonnées pas ds les bornes
df["coordonnees_dans_enveloppe_france"] = (df["longitude"].between(-5.5, 10.0) & df["latitude"].between(41.0, 51.5))

print("Lignes avec coordonnées manquantes :", int((df["longitude"].isna() | df["latitude"].isna()).sum()))
print("Lignes hors de la France métropolitaine :", int((~df["coordonnees_dans_enveloppe_france"]).sum()))

In [ ]:
liste_communes = df["commune_analyse"].unique()
resultats_communes = []

for commune in liste_communes:
    if pd.isna(commune):
        donnees_commune = df[df["commune_analyse"].isna()]
    else:
        donnees_commune = df[df["commune_analyse"] == commune]

    nombre_enregistrements = len(donnees_commune)
    nombre_stations = donnees_commune["id_station_itinerance"].nunique()
    nombre_pdc_identifies = donnees_commune["id_pdc_itinerance"].nunique()

    puissance_moyenne = donnees_commune["puissance_kW"].mean()
    puissance_mediane = donnees_commune["puissance_kW"].median()
    puissance_totale = donnees_commune["puissance_kW"].sum(min_count=1)

    nombre_voie_publique = (donnees_commune["implantation_categorie"] == "Voie publique").sum()
    if len(donnees_commune) > 0:
        part_voie_publique = (nombre_voie_publique / len(donnees_commune)) * 100
    else:
        part_voie_publique = float("nan")

    nombre_autre = (donnees_commune["implantation_categorie"] == "Autre").sum()
    if len(donnees_commune) > 0:
        part_autre = (nombre_autre / len(donnees_commune)) * 100
    else:
        part_autre = float("nan")

    longitude_moyenne = donnees_commune["longitude"].mean()
    latitude_moyenne = donnees_commune["latitude"].mean()

    resultat = {
        "commune_analyse": commune,
        "nombre_enregistrements": nombre_enregistrements,
        "nombre_stations": nombre_stations,
        "nombre_pdc_identifies": nombre_pdc_identifies,
        "puissance_moyenne_kW": puissance_moyenne,
        "puissance_mediane_kW": puissance_mediane,
        "puissance_totale_kW": puissance_totale,
        "part_voie_publique": part_voie_publique,
        "part_autre": part_autre,
        "longitude_moyenne": longitude_moyenne,
        "latitude_moyenne": latitude_moyenne
    }
    resultats_communes.append(resultat)
indicateurs_commune = pd.DataFrame(resultats_communes)
indicateurs_commune = indicateurs_commune.sort_values("nombre_pdc_identifies",ascending=False)

display(indicateurs_commune.head())
print("Nombre de communes / groupes :", len(indicateurs_commune))


### Comparer les types d'implantation et les vitesses
Ces tableaux résument le nombre d'enregistrements et les puissances selon la catégorie d'implantation et la vitesse de recharge.

In [ ]:
#Tableau de synthèse du nombre de points de charge par type d'implantation
tableau_implantations = (df.groupby("implantation_categorie", dropna=False).agg(
          nombre_enregistrements=("implantation_categorie", "size"),
          puissance_moyenne_kW=("puissance_kW", "mean"),
          puissance_mediane_kW=("puissance_kW", "median")
      )
      .sort_values("nombre_enregistrements", ascending=False)
      .reset_index()
      .rename(columns={
          "implantation_categorie": "Type d'implantation",
          "nombre_enregistrements": "Nombre de points de charge enregistrés",
          "puissance_moyenne_kW": "Puissance moyenne (kW)",
          "puissance_mediane_kW": "Puissance médiane (kW)"
      })
)

display(tableau_implantations.round(2))

# Tableau par catégorie de vitesse
tableau_vitesses = (df["categorie_vitesse"].value_counts(dropna=False).rename_axis("Catégorie de vitesse de recharge").reset_index(name="Nombre de points de charge enregistrés"))
display(tableau_vitesses)

In [ ]:
# Graphique 1 : nombre d'enregistrements par implantation
compte_implantations = df["implantation_categorie"].value_counts().reindex(ordre_implantations, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 6))
compte_implantations.plot(kind="bar", ax=ax, color=PALETTE_PASTEL[:len(compte_implantations)])
ax.set_title("Répartition des points de charge par type d'implantation")
ax.set_xlabel("Type d'implantation")
ax.set_ylabel("Nombre d'enregistrements")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Graphique 2 : catégories de vitesse
compte_vitesses = df["categorie_vitesse"].value_counts()
fig, ax = plt.subplots(figsize=(9, 5))
compte_vitesses.plot(kind="bar", ax=ax, color=PALETTE_PASTEL[:len(compte_vitesses)])
ax.set_title("Répartition des points de charge par vitesse de recharge")
ax.set_xlabel("Catégorie de vitesse")
ax.set_ylabel("Nombre d'enregistrements")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Graphique 3 : communes ayant le plus de points de charge identifiés
top_communes = indicateurs_commune.nlargest(15, "nombre_pdc_identifies")
top_communes = top_communes.sort_values("nombre_pdc_identifies")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_communes["commune_analyse"], top_communes["nombre_pdc_identifies"],
        color=PALETTE_PASTEL[:len(top_communes)] if len(top_communes) <= len(PALETTE_PASTEL)
        else [PALETTE_PASTEL[i % len(PALETTE_PASTEL)] for i in range(len(top_communes))])
ax.set_title("15 communes avec le plus de points de charge identifiés")
ax.set_xlabel("Nombre d'identifiants de points de charge distincts")
ax.set_ylabel("Commune")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# Graphique 4 : distribution de la puissance (sans supprimer les valeurs extrêmes)
# Pour garder un graphique lisible, l'axe est limité au 99e percentile ; aucune donnée n'est supprimée.
puissances = df.loc[df["puissance_kW"].notna() & (df["puissance_kW"] > 0), "puissance_kW"]
if len(puissances):
    limite_visuelle = puissances.quantile(0.99)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(puissances.clip(upper=limite_visuelle), bins=30, color=PALETTE_PASTEL[2], edgecolor="white")
    ax.set_title(f"Distribution des puissances en kW (valeurs affichées jusqu'au 99e percentile : {limite_visuelle:.1f} kW)")
    ax.set_xlabel("Puissance nominale (kW), valeurs au-delà du seuil regroupées dans la dernière classe")
    ax.set_ylabel("Nombre d'enregistrements")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
#import requests

#url = "https://etalab-datasets.geo.data.gouv.fr/contours-administratifs/latest/geojson/epci-100m.geojson"

#r = requests.get(url, timeout=60)

#print("Code HTTP :", r.status_code)
#print("Taille du fichier :", len(r.content), "octets")

#if r.status_code == 200:
    #with open("epci-100m.geojson", "wb") as f:
        #f.write(r.content)
    #print("Fichier téléchargé avec succès !")
#else:
    print(r.text[:500])

### Installer les bibliothèques de cartographie
Cette commande installe les bibliothèques nécessaires à la carte interactive. Elle doit être exécutée dans Jupyter si elles ne sont pas déjà installées.

In [ ]:
%pip install geopandas folium branca

In [ ]:
# CARTE INTERACTIVE DES BORNES PAR EPCI
import json
import requests
import geopandas as gpd
import folium
import branca.colormap as cm
from pathlib import Path
from branca.element import MacroElement
from jinja2 import Template

fichier_epci = Path("epci-100m.geojson")
epci = gpd.read_file(fichier_epci)

if "code_epci" not in epci.columns and "code" in epci.columns:
    epci = epci.rename(columns={"code": "code_epci"})

if "nom_epci" not in epci.columns and "nom" in epci.columns:
    epci = epci.rename(columns={"nom": "nom_epci"})

if "code_epci" not in epci.columns:
    raise ValueError("Colonne code_epci introuvable dans le fichier.")

if "nom_epci" not in epci.columns:
    epci["nom_epci"] = "EPCI " + epci["code_epci"].astype(str)

epci["code_epci"] = epci["code_epci"].astype(str).str.strip()

#Préparation des données de bornes
donnees = df.copy()
donnees["code_insee_commune"] = donnees["code_insee_commune"].astype("string").str.strip()
donnees = donnees.dropna(subset=["code_insee_commune"])

#convertir le nombre de pdc en valeur numérique
donnees["nbre_pdc"] = pd.to_numeric(donnees["nbre_pdc"], errors="coerce").fillna(0)
donnees["nbre_pdc"] = donnees["nbre_pdc"].clip(lower=0)

url_communes = "https://geo.api.gouv.fr/communes?fields=code,nom,codeEpci&format=json"
reponse = requests.get(url_communes, timeout=60)
reponse.raise_for_status()
communes = pd.DataFrame(reponse.json())

if "codeEpci" not in communes.columns:
    raise ValueError("La réponse de l'API ne contient pas la colonne codeEpci.")

communes = communes.rename(columns={
    "code": "code_insee_commune",
    "nom": "nom_commune",
    "codeEpci": "code_epci"
})

communes["code_insee_commune"] = communes["code_insee_commune"].astype(str).str.strip()
communes["code_epci"] = communes["code_epci"].astype("string").str.strip()

#on associe à chaque ligne de données le code EPCI de sa commune
donnees = donnees.merge(
    communes[["code_insee_commune", "code_epci"]],
    on="code_insee_commune",
    how="left"
)
donnees = donnees.dropna(subset=["code_epci"])
donnees["code_epci"] = donnees["code_epci"].astype(str)

if "id_station_itinerance" in donnees.columns:
    donnees["id_station_analyse"] = donnees["id_station_itinerance"].astype(str)
else:
    donnees["id_station_analyse"] = ""

if "id_station_local" in donnees.columns:
    masque_vide = donnees["id_station_analyse"].eq("") | donnees["id_station_analyse"].isna()
    donnees.loc[masque_vide, "id_station_analyse"] = donnees["id_station_local"].astype(str)

masque_vide = donnees["id_station_analyse"].eq("") | donnees["id_station_analyse"].isna() | donnees["id_station_analyse"].str.strip().eq("")

if masque_vide.any():
    nom = donnees["nom_station"].astype(str) if "nom_station" in donnees.columns else ""
    adresse = donnees["adresse_station"].astype(str) if "adresse_station" in donnees.columns else ""
    donnees.loc[masque_vide, "id_station_analyse"] = nom + " | " + adresse

# Une ligne par station pour ne pas additionner plusieurs fois ses points
stations_uniques = donnees.drop_duplicates(subset=["code_epci", "id_station_analyse"]).copy()

nb_stations = stations_uniques.groupby("code_epci").size().rename("nombre_stations")
points_par_station = donnees.sort_values("nbre_pdc", ascending=False).drop_duplicates(subset=["code_epci", "id_station_analyse"])
nb_points = points_par_station.groupby("code_epci")["nbre_pdc"].sum().rename("nombre_points_charge")
indicateurs = pd.concat([nb_stations, nb_points], axis=1).fillna(0)

url_population = "https://geo.api.gouv.fr/communes?fields=code,population&format=json"
reponse_pop = requests.get(url_population, timeout=60)
reponse_pop.raise_for_status()
population_communes = pd.DataFrame(reponse_pop.json())
population_communes = population_communes.rename(columns={
    "code": "code_insee_commune",
    "population": "population"
})

population_communes["code_insee_commune"] = population_communes["code_insee_commune"].astype(str).str.strip()
population_communes["population"] = pd.to_numeric(population_communes["population"], errors="coerce")

population_communes = population_communes.merge(
    communes[["code_insee_commune", "code_epci"]],
    on="code_insee_commune",
    how="left"
)

population_epci = population_communes.dropna(subset=["code_epci"]).groupby("code_epci")["population"].sum(min_count=1).rename("population")
indicateurs = indicateurs.join(population_epci, how="left")

#calcul des densités et ratios
epci_metrique = epci.to_crs(epsg=2154)
epci["surface_km2"] = epci_metrique.geometry.area / 1_000_000
epci["code_epci"] = epci["code_epci"].astype(str)

carte_data = epci.merge(
    indicateurs.reset_index().rename(columns={"index": "code_epci"}),
    on="code_epci",
    how="left"
)

for colonne in ["nombre_stations", "nombre_points_charge"]:
    if colonne not in carte_data.columns:
        carte_data[colonne] = 0
    carte_data[colonne] = carte_data[colonne].fillna(0)

if "population" not in carte_data.columns:
    carte_data["population"] = np.nan

carte_data["stations_pour_1000_habitants"] = (
    carte_data["nombre_stations"] / carte_data["population"].replace(0, np.nan) * 1000
)
carte_data["points_pour_1000_habitants"] = (
    carte_data["nombre_points_charge"] / carte_data["population"].replace(0, np.nan) * 1000
)
carte_data["stations_par_km2"] = (
    carte_data["nombre_stations"] / carte_data["surface_km2"].replace(0, np.nan)
)
carte_data["points_par_km2"] = (
    carte_data["nombre_points_charge"] / carte_data["surface_km2"].replace(0, np.nan)
)

carte_data = carte_data.replace([np.inf, -np.inf], np.nan)

#construction de la carte interactive
centre = [46.6, 2.5]
carte = folium.Map(location=centre, zoom_start=6, tiles=None, control_scale=True)

indicateurs_carte = {
    "Nombre de stations": "nombre_stations",
    "Nombre de points de charge": "nombre_points_charge",
    "Stations pour 1 000 habitants": "stations_pour_1000_habitants",
    "Points de charge pour 1 000 habitants": "points_pour_1000_habitants",
    "Stations par km²": "stations_par_km2",
    "Points de charge par km²": "points_par_km2",
}

couleurs = ["#a50026", "#f46d43", "#fee08b", "#a6d96a", "#006837"]
libelles = ["Très faible", "Faible", "Intermédiaire", "Élevé", "Très élevé"]

for nom_indicateur, colonne in indicateurs_carte.items():
    valeurs = pd.to_numeric(carte_data[colonne], errors="coerce")
    valeurs_valides = valeurs.dropna()

    if valeurs_valides.empty:
        print(f"Indicateur ignoré : {nom_indicateur}")
        continue

    seuils = valeurs_valides.quantile([0.2, 0.4, 0.6, 0.8]).tolist()
    seuils = sorted(set(seuils))

    def classer_valeur(valeur):
        if pd.isna(valeur):
            return None
        return int(np.searchsorted(seuils, valeur, side="left"))

    couche = folium.FeatureGroup(name=nom_indicateur, show=(nom_indicateur == "Nombre de stations"))

    for _, ligne in carte_data.iterrows():
        valeur = ligne[colonne]
        nom = str(ligne["nom_epci"])
        code = str(ligne["code_epci"])

        if pd.isna(valeur):
            couleur = "#d9d9d9"
            categorie = "Donnée indisponible"
            valeur_affichee = "Non disponible"
        else:
            indice = classer_valeur(valeur)
            couleur = couleurs[indice]
            categorie = libelles[indice]
            valeur_affichee = f"{valeur:,.2f}".replace(",", " ")

        popup_html = f"""
        <div style="width:270px">
            <b>{nom}</b><br>
            Code EPCI : {code}<hr>
            <b>Indicateur :</b> {nom_indicateur}<br>
            <b>Valeur :</b> {valeur_affichee}<br>
            <b>Catégorie :</b> {categorie}
        </div>
        """

        folium.GeoJson(
            data={
                "type": "Feature",
                "geometry": ligne.geometry.__geo_interface__,
                "properties": {}
            },
            style_function=lambda feature, couleur=couleur: {
                "fillColor": couleur,
                "color": "#555555",
                "weight": 0.5,
                "fillOpacity": 0.85,
            },
            tooltip=folium.Tooltip(f"{nom} — {nom_indicateur} : {valeur_affichee}"),
            popup=folium.Popup(popup_html, max_width=320),
        ).add_to(couche)

    couche.add_to(carte)

    lignes_legende = ""
    for i, couleur in enumerate(couleurs):
        if i < len(seuils):
            borne = seuils[i]
            texte = f"≤ {borne:.2f}".replace(".", ",")
        else:
            texte = f"> {seuils[-1]:.2f}".replace(".", ",")

        lignes_legende += f"""
        <div style="display:flex;align-items:center;margin:4px 0">
            <span style="display:inline-block;width:18px;height:14px;background:{couleur};margin-right:8px;border:1px solid #777"></span>
            <span>{libelles[i]} ({texte})</span>
        </div>
        """

    legende = MacroElement()
    legende._template = Template(f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        bottom: 35px;
        left: 35px;
        z-index: 9999;
        background: white;
        padding: 12px;
        border: 2px solid #777;
        border-radius: 5px;
        font-size: 12px;
        max-width: 250px;
        ">
        <b>{nom_indicateur}</b>
        <hr style="margin:6px 0">
        {lignes_legende}
        <div style="margin-top:8px;color:#666">
            Gris : donnée indisponible
        </div>
    </div>
    {{% endmacro %}}
    """)
    carte.get_root().add_child(legende)

folium.LayerControl(collapsed=False).add_to(carte)

minx, miny, maxx, maxy = carte_data.total_bounds
carte.fit_bounds([[miny, minx], [maxy, maxx]])

display(carte)

In [ ]:
colonnes_analyse = ["code_epci","nom_epci","population","surface_km2","nombre_stations","nombre_points_charge","stations_pour_1000_habitants","points_pour_1000_habitants","stations_par_km2","points_par_km2"]

analyse_epci = carte_data[[c for c in colonnes_analyse if c in carte_data.columns]].copy()

# Conversion en nombres pour éviter les erreurs de calcul
colonnes_numeriques = ["population","surface_km2","nombre_stations","nombre_points_charge","stations_pour_1000_habitants","points_pour_1000_habitants","stations_par_km2","points_par_km2"]

for colonne in colonnes_numeriques:
    if colonne in analyse_epci.columns:
        analyse_epci[colonne] = pd.to_numeric(
            analyse_epci[colonne],
            errors="coerce"
        )

# Contrôler la qualité des données
print("Nombre total d'EPCI :", len(analyse_epci))

print(
    "EPCI avec population connue :",
    analyse_epci["population"].notna().sum()
)

In [ ]:
reference = analyse_epci[(analyse_epci["population"] > 0) & analyse_epci["points_pour_1000_habitants"].notna()].copy()

if reference.empty:
    raise ValueError(
        "Impossible de calculer la référence : "
        "vérifie les données de population et de points de charge."
    )

mediane_points_1000 = (reference["points_pour_1000_habitants"].median())
mediane_stations_1000 = (reference["stations_pour_1000_habitants"].median())

mediane_points_km2 = (
    reference["points_par_km2"].median()
)

In [ ]:
#écart la médiane
analyse_epci["ecart_median_points_1000"] = (analyse_epci["points_pour_1000_habitants"] - mediane_points_1000)

analyse_epci["ecart_relatif_median_pct"] = ((analyse_epci["points_pour_1000_habitants"] / mediane_points_1000) - 1) * 100 if mediane_points_1000 > 0 else np.nan

#on identifie les territoires sous la médiane
analyse_epci["sous_median_equipement"] = (analyse_epci["points_pour_1000_habitants"]< mediane_points_1000)

#classement du moins équipé au mieux équipé
classement_sous_equipement = (analyse_epci[(analyse_epci["population"] > 0) & analyse_epci["points_pour_1000_habitants"].notna()]
    .sort_values("points_pour_1000_habitants",ascending=True)
    .reset_index(drop=True)
)

classement_sous_equipement.insert(0,"rang_sous_equipement",range(1, len(classement_sous_equipement) + 1))

# Afficher les 5 EPCI les moins équipés par habitant
colonnes_affichage = [
    "rang_sous_equipement",
    "nom_epci",
    "population",
    "nombre_stations",
    "nombre_points_charge",
    "points_pour_1000_habitants",
    "ecart_relatif_median_pct"
]

display(classement_sous_equipement[colonnes_affichage].head().round(3))

In [ ]:
#Reco france métropolitaine et outre mers
# On se limite ici aux DROM : Guadeloupe, Martinique, Guyane, La Réunion et Mayotte.

codes_outre_mer = {
    "971",  # Guadeloupe
    "972",  # Martinique
    "973",  # Guyane
    "974",  # La Réunion
    "976"   # Mayotte
}

communes_zones = communes[["code_insee_commune", "code_epci"]].drop_duplicates().copy()
communes_zones["code_insee_commune"] = (communes_zones["code_insee_commune"].astype(str).str.strip())
communes_zones["zone"] = (communes_zones["code_insee_commune"].str[:3].apply(lambda code: ("Outre-mer"
            if code in codes_outre_mer
            else "À vérifier"
        )
    )
)

# Déterminer la zone d'un EPCI à partir des communes qui le composent.
zones_par_epci = (communes_zones.groupby("code_epci")["zone"].agg(lambda x: list(set(x.dropna()))).reset_index())

def determiner_zone(zones):
    zones_valides = [z for z in zones if z in ["Outre-mer", "À vérifier"]]

    if len(zones_valides) == 1:
        return zones_valides[0]

    return "À vérifier"

zones_par_epci["zone"] = zones_par_epci["zone"].apply(determiner_zone)


# Affectation provisoire des EPCI :
# - Outre-mer si toutes les communes rattachées appartiennent à un DROM
# - France métropolitaine pour les codes départementaux classiques
codes_metropole = {
    f"{i:02d}" for i in range(1, 96)
} - {"20"}

def zone_depuis_code(code_insee):
    code = str(code_insee).strip()

    if code[:3] in codes_outre_mer:
        return "Outre-mer"

    if code[:2] in codes_metropole:
        return "France métropolitaine"

    if code.startswith("2A") or code.startswith("2B"):
        return "France métropolitaine"

    return "À vérifier"

communes_zones["zone"] = (communes_zones["code_insee_commune"].apply(zone_depuis_code))

zones_par_epci = (communes_zones.groupby("code_epci")["zone"].agg(lambda x: list(set(x.dropna()))).reset_index())
zones_par_epci["zone"] = zones_par_epci["zone"].apply(lambda zones: zones[0] if len(zones) == 1 else "À vérifier")

# Ajouter la zone au tableau d'analyse
analyse_epci = analyse_epci.drop(columns=["zone"],errors="ignore").merge(
    zones_par_epci[["code_epci", "zone"]],
    on="code_epci",
    how="left"
)

print("Répartition des EPCI par zone :")
for zone, nombre in analyse_epci["zone"].value_counts(dropna=False).items():
    print(f"- {zone} : {nombre}")

In [ ]:
#classement par zone
population_minimale = 10000
base_classement = analyse_epci[(analyse_epci["population"] >= population_minimale) & (analyse_epci["population"].notna()) & (analyse_epci["points_pour_1000_habitants"].notna())].copy()

# Séparer métropole et outre-mer
classement_metropole = (base_classement[base_classement["zone"] == "France métropolitaine"].sort_values("points_pour_1000_habitants", ascending=True).reset_index(drop=True))
classement_outre_mer = (base_classement[base_classement["zone"] == "Outre-mer"].sort_values("points_pour_1000_habitants", ascending=True).reset_index(drop=True))

classement_metropole.insert(0,"rang_zone",range(1, len(classement_metropole) + 1))
classement_outre_mer.insert(0,"rang_zone",range(1, len(classement_outre_mer) + 1))

colonnes_resultats = [
    "rang_zone",
    "nom_epci",
    "population",
    "nombre_stations",
    "nombre_points_charge",
    "points_pour_1000_habitants",
    "points_par_km2"
]

print("LES 5 EPCI LES MOINS ÉQUIPÉS EN FRANCE MÉTROPOLITAINE")
display(classement_metropole[colonnes_resultats].head(5).round(3))

print("LES 5 EPCI LES MOINS ÉQUIPÉS EN OUTRE-MER")
display(classement_outre_mer[colonnes_resultats].head(5).round(3))

In [ ]:
# ============================================================
# 11.3 — ESTIMATION DES BESOINS THÉORIQUES
# ============================================================

reference_nationale = base_classement[
    base_classement["zone"].isin(
        ["France métropolitaine", "Outre-mer"]
    )
]

mediane_nationale = (
    reference_nationale["points_pour_1000_habitants"].median()
)

def ajouter_besoins(tableau):
    tableau = tableau.copy()

    tableau["points_cible_median"] = np.ceil(
        tableau["population"] * mediane_nationale / 1000
    )

    tableau["points_supplementaires_theoriques"] = (
        tableau["points_cible_median"]
        - tableau["nombre_points_charge"]
    ).clip(lower=0)

    tableau["points_supplementaires_theoriques"] = (
        tableau["points_supplementaires_theoriques"]
        .fillna(0)
        .astype(int)
    )

    return tableau

classement_metropole = ajouter_besoins(classement_metropole)
classement_outre_mer = ajouter_besoins(classement_outre_mer)

top5_metropole = classement_metropole.head(5).copy()
top5_outre_mer = classement_outre_mer.head(5).copy()

colonnes_recommandations = [
    "rang_zone",
    "nom_epci",
    "population",
    "nombre_points_charge",
    "points_pour_1000_habitants",
    "points_cible_median",
    "points_supplementaires_theoriques"
]

print("RECOMMANDATIONS — FRANCE MÉTROPOLITAINE")

display(
    top5_metropole[colonnes_recommandations].round(3)
)

print("RECOMMANDATIONS — OUTRE-MER")

display(
    top5_outre_mer[colonnes_recommandations].round(3)
)

In [ ]:
fichier_carte = dossier_exports / "carte_epci_bornes_irve.html"
carte.save(str(fichier_carte))
print("Carte interactive enregistrée :", fichier_carte.resolve())